# LLM Inference and Analysis from ShareGPT Data (using RAG)

## 1. Setup and Load Data

In [ ]:
import json
import os
import requests
# Removed base64, PIL, io as direct encoding is not needed for RAG upload
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from dotenv import load_dotenv
import time
import re

# --- Configuration ---
conversation_file = '../food_conversations.json' # Assumes file is in the parent directory
hpi_api_base = "https://chat.hpi-sci.de/api" # Base URL for HPI API
# Model identifier for chat completions 
model_identifier = "openai/mistralai/Pixtral-12B-2409" 
output_results_file = 'llm_rag_inference_results.json' # File to save results (updated name)
max_tokens_response = 500 # Max tokens expected in the LLM response JSON
# -- End Configuration --

# Load API Key from .env file (assumed to be in parent directory)
dotenv_path = '../.env'
load_dotenv(dotenv_path=dotenv_path)
api_key = os.getenv('API_KEY')

if not api_key:
    print(f"Error: API_KEY not found in {os.path.abspath(dotenv_path)}")
    # Add manual input or raise error if needed
else:
    print("API Key loaded successfully.")
    
# Load the ShareGPT conversation data
if os.path.exists(conversation_file):
    try:
        with open(conversation_file, 'r', encoding='utf-8') as f:
            conversations = json.load(f)
        print(f"Loaded {len(conversations)} conversations from {conversation_file}")
    except Exception as e:
        print(f"Error loading {conversation_file}: {e}")
        conversations = [] # Ensure conversations is defined
else:
    print(f"Error: Conversation file not found at {os.path.abspath(conversation_file)}")
    conversations = []

## 2. Helper Functions (File Upload, API Call, JSON Parsing)

In [ ]:
def upload_file_to_hpi(image_path, api_key):
    """Uploads a file to the HPI/OpenWebUI file endpoint."""
    if not image_path or not os.path.exists(image_path):
        print(f"Error: Image path for upload not found or empty: {image_path}")
        return None
        
    upload_endpoint = f"{hpi_api_base}/v1/files/"
    headers = {
        'Authorization': f'Bearer {api_key}',
        'Accept': 'application/json' # Important for getting JSON response
    }
    
    try:
        with open(image_path, 'rb') as f:
            files = {'file': (os.path.basename(image_path), f)}
            response = requests.post(upload_endpoint, headers=headers, files=files)
            response.raise_for_status()
            
            response_data = response.json()
            # Assuming the response contains an 'id' field for the uploaded file
            file_id = response_data.get('id') 
            if file_id:
                print(f"  Successfully uploaded {os.path.basename(image_path)}, got file_id: {file_id}")
                return file_id
            else:
                print(f"Error: File ID not found in upload response for {os.path.basename(image_path)}")
                print(f"Response: {response_data}")
                return None
                
    except requests.exceptions.RequestException as e:
        print(f"Error uploading file {os.path.basename(image_path)}: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response status: {e.response.status_code}")
            print(f"Response text: {e.response.text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred during file upload for {os.path.basename(image_path)}: {e}")
        return None

def extract_json_from_response(response_text):
    """Extracts the first JSON block from the LLM's response markdown."""
    if not response_text or not isinstance(response_text, str):
        print("Warning: Invalid response text received for JSON extraction.")
        return None
    # Regex to find JSON block enclosed in ```json ... ```
    match = re.search(r'```json\s*({.*?})\s*```', response_text, re.DOTALL)
    if match:
        json_str = match.group(1)
        try:
            return json.loads(json_str)
        except json.JSONDecodeError as e:
            print(f"Error decoding extracted JSON: {e}")
            print(f"Extracted string: {json_str}")
            return None
    else:
        # Fallback: try parsing the whole text if no markdown block found
        try:
            return json.loads(response_text)
        except json.JSONDecodeError:
            print("Warning: No JSON block found in response, and response is not valid JSON itself.")
            print(f"Response text: {response_text[:500]}..." ) # Print snippet
            return None

def call_hpi_llm_rag(prompt_text, file_id, api_key):
    """Calls the HPI chat completions endpoint referencing an uploaded file ID."""
    headers = {
        "Authorization": f"Bearer {api_key}",
        "Content-Type": "application/json"
    }
    
    # Construct the payload 
    payload = {
        "model": model_identifier,
        "messages": [
            {
                "role": "user",
                "content": prompt_text # Just the text prompt here
            }
        ],
        "max_tokens": max_tokens_response
    }
    
    # Add file reference if file_id is available
    if file_id:
        payload["files"] = [
            {"type": "file", "id": file_id}
        ]
        
    # Use the standard chat completions endpoint path
    endpoint = f"{hpi_api_base}/v1/chat/completions" 
    
    try:
        response = requests.post(endpoint, headers=headers, json=payload)
        response.raise_for_status() # Raise an exception for bad status codes (4xx or 5xx)
        
        response_data = response.json()
        # Extract content from the first choice
        if response_data.get('choices') and len(response_data['choices']) > 0:
            message_content = response_data['choices'][0].get('message', {}).get('content')
            return message_content
        else:
            print("Error: No choices found in LLM response.")
            print(f"Response: {response_data}")
            return None
            
    except requests.exceptions.RequestException as e:
        print(f"Error calling HPI LLM API: {e}")
        if hasattr(e, 'response') and e.response is not None:
            print(f"Response status: {e.response.status_code}")
            print(f"Response text: {e.response.text}")
        return None
    except Exception as e:
        print(f"An unexpected error occurred during API call: {e}")
        return None

## 3. Run Inference Loop (RAG Workflow)

In [ ]:
inference_results = []
start_time_inference = time.time()

if not conversations:
    print("No conversations loaded. Cannot run inference.")
elif not api_key:
    print("API Key not available. Cannot run inference.")
else:
    print(f"Starting inference for {len(conversations)} conversations using RAG...")
    for i, conv in enumerate(conversations):
        print(f"Processing conversation {i+1}/{len(conversations)}...")
        
        # Extract data from conversation
        file_id = None # Initialize file_id
        try:
            user_message = conv['messages'][0]['content']
            assistant_message = conv['messages'][1]['content']
            image_path = conv['images'][0] if conv.get('images') else None
            entry_id = os.path.basename(image_path) if image_path else f'conv_{i}'
            
            # Clean user prompt (remove <image> placeholder)
            prompt_text = user_message.replace('<image>', '').strip()
            
            # Get ground truth JSON from assistant message
            ground_truth_json = extract_json_from_response(assistant_message)
            if not ground_truth_json:
                print(f"  Warning: Could not parse ground truth JSON for {entry_id}. Skipping.")
                continue
            
        except (IndexError, KeyError, TypeError) as e:
             print(f"  Error extracting data from conversation {i}: {e}. Skipping.")
             continue
             
        # Upload image if path exists
        if image_path:
            print(f"  Uploading image {os.path.basename(image_path)}..." )
            file_id = upload_file_to_hpi(image_path, api_key)
            if not file_id:
                 print(f"  Warning: Failed to upload image for {entry_id}. Proceeding without image context.")
        else:
            print("  No image path found in conversation data.")
        
        # Call the LLM (with or without file_id)
        print(f"  Calling LLM for {entry_id} (file_id: {file_id})..." )
        llm_response_content = call_hpi_llm_rag(prompt_text, file_id, api_key)
        
        # Process response
        if llm_response_content:
            predicted_json = extract_json_from_response(llm_response_content)
            if predicted_json:
                print(f"  Successfully received and parsed prediction for {entry_id}")
                inference_results.append({
                    "id": entry_id,
                    "image_path": image_path,
                    "used_file_id": file_id, # Store the file_id used
                    "prediction": predicted_json,
                    "ground_truth": ground_truth_json
                })
            else:
                print(f"  Warning: Failed to parse JSON prediction for {entry_id}")
                inference_results.append({
                    "id": entry_id,
                    "image_path": image_path,
                    "used_file_id": file_id,
                    "prediction": {"error": "Failed to parse response", "raw_response": llm_response_content},
                    "ground_truth": ground_truth_json 
                })
        else:
            print(f"  Warning: No valid response received from LLM for {entry_id}")
            inference_results.append({
                "id": entry_id,
                "image_path": image_path,
                "used_file_id": file_id,
                "prediction": {"error": "No response from LLM"},
                "ground_truth": ground_truth_json 
            })
            
        # Optional delay between API calls
        # time.sleep(1) 
        
end_time_inference = time.time()
total_inference_time = end_time_inference - start_time_inference
print(f"\nFinished inference loop in {total_inference_time:.2f} seconds.")
print(f"Processed {len(inference_results)} entries.")

# Save results
if inference_results:
    try:
        with open(output_results_file, 'w', encoding='utf-8') as f:
            json.dump(inference_results, f, indent=4)
        print(f"Inference results saved to {os.path.abspath(output_results_file)}")
    except Exception as e:
        print(f"Error saving results to {output_results_file}: {e}")

## 4. Analyze Results

In [ ]:
# This cell remains largely the same, as it analyzes the saved results structure
# which still contains 'prediction' and 'ground_truth' keys.
def calculate_total_returned(data_json):
    """Calculates total returned weight from prediction or ground truth JSON."""
    total = 0.0
    if not isinstance(data_json, dict) or 'ingredients' not in data_json:
        return None # Indicate parsing error or invalid structure
        
    for item in data_json.get('ingredients', []):
        try:
            # Ground truth uses 'returned_weight', predictions might use the same
            returned = item.get('returned_weight', 0.0)
            total += float(returned)
        except (ValueError, TypeError):
            # Ignore items where returned weight is not a valid number
            pass 
    return total

# Load results if they weren't computed in this session
if 'inference_results' not in locals() or not inference_results:
    if os.path.exists(output_results_file):
        print(f"Loading results from {output_results_file}")
        try:
            with open(output_results_file, 'r', encoding='utf-8') as f:
                inference_results = json.load(f)
        except Exception as e:
            print(f"Error loading results file: {e}")
            inference_results = []
    else:
        print("Error: No inference results available to analyze.")
        inference_results = []

# Prepare data for analysis
analysis_data = []
if inference_results:
    for result in inference_results:
        pred_total = calculate_total_returned(result.get('prediction'))
        gt_total = calculate_total_returned(result.get('ground_truth'))
        
        # Only include entries where both totals could be calculated
        if pred_total is not None and gt_total is not None:
            analysis_data.append({
                'id': result.get('id'),
                'predicted_total_returned': pred_total,
                'actual_total_returned': gt_total
            })
        else:
             print(f"Skipping entry {result.get('id')} due to missing/invalid prediction or ground truth total.")
             
# Create DataFrame
if analysis_data:
    analysis_df = pd.DataFrame(analysis_data)
    print(f"\nCreated analysis DataFrame with {len(analysis_df)} valid entries.")
    display(analysis_df.head())
    
    # Calculate Metrics
    if not analysis_df.empty:
        mae = np.mean(np.abs(analysis_df['predicted_total_returned'] - analysis_df['actual_total_returned']))
        mse = np.mean((analysis_df['predicted_total_returned'] - analysis_df['actual_total_returned'])**2)
        rmse = np.sqrt(mse)
        
        # Avoid division by zero for MAPE
        analysis_df_filtered = analysis_df[analysis_df['actual_total_returned'] != 0]
        if not analysis_df_filtered.empty:
             mape = np.mean(np.abs((analysis_df_filtered['predicted_total_returned'] - analysis_df_filtered['actual_total_returned']) / analysis_df_filtered['actual_total_returned'])) * 100
        else:
             mape = float('nan') # Assign NaN if no non-zero ground truth values exist
             
        print(f"\nEvaluation Metrics (Total Returned Weight):")
        print(f"MAE:  {mae:.2f}g")
        print(f"RMSE: {rmse:.2f}g")
        print(f"MAPE: {mape:.2f}%")

        # Plotting
        plt.figure(figsize=(10, 6))
        sns.scatterplot(data=analysis_df, x='actual_total_returned', y='predicted_total_returned')
        
        # Add identity line (perfect prediction)
        if not analysis_df.empty:
             max_val = max(analysis_df['actual_total_returned'].max(), analysis_df['predicted_total_returned'].max())
             min_val = min(analysis_df['actual_total_returned'].min(), analysis_df['predicted_total_returned'].min())
             # Add small buffer to prevent line ending exactly at points
             buffer = (max_val - min_val) * 0.05 
             plt.plot([min_val - buffer, max_val + buffer], [min_val - buffer, max_val + buffer], color='red', linestyle='--', label='Perfect Prediction')
             plt.xlim(min_val - buffer, max_val + buffer) # Adjust plot limits
             plt.ylim(min_val - buffer, max_val + buffer)
        else: 
             # Fallback if dataframe is empty, plot a simple line
             plt.plot([0, 1], [0, 1], color='red', linestyle='--', label='Perfect Prediction') 
        
        plt.title('Predicted vs. Actual Total Returned Weight (RAG via API)')
        plt.xlabel('Actual Total Returned (g) - From food_conversations.json')
        plt.ylabel('Predicted Total Returned (g) - From LLM')
        plt.legend()
        plt.grid(True)
        plt.show()
else:
    print("No valid data available for analysis after processing results.")